# EXAONE 3.5 7.8B Colab Server

This notebook runs `LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct` on Google Colab GPU with 4-bit quantization and exposes an OpenAI-compatible API for the local `K-Safety Law RAG` CLI/Web UI.

## Flow
1. Set Colab runtime to GPU.
2. Run cells from top to bottom.
3. Copy the ngrok URL printed at the end.
4. Set local `.env` with `LLM_API_BASE=<ngrok-url>/v1`.
5. Set local `.env` with `LLM_MODEL=LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct`.
6. Run `python web_app.py` or `python cli.py` locally.

## Local `.env` example
```env
LLM_PROVIDER=remote_openai
LLM_MODEL=LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct
LLM_API_BASE=https://xxxxx.ngrok-free.app/v1
LLM_API_KEY=dummy
```


In [ ]:
# 1. Check GPU
!nvidia-smi


In [ ]:
# 2. Install dependencies
# EXAONE remote code currently expects the newer Transformers masking API.
# After this cell finishes, restart the Colab runtime once, then continue from cell 3.
!pip uninstall -y transformers accelerate
!pip install -q git+https://github.com/huggingface/transformers.git "accelerate>=1.0.0" bitsandbytes fastapi uvicorn pyngrok nest_asyncio huggingface_hub requests

print("IMPORTANT: Runtime > Restart runtime, then continue from cell 3.")


In [ ]:
# 3. Check runtime
import transformers
import torch

print("transformers", transformers.__version__)
print("torch", torch.__version__)
print("cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu", torch.cuda.get_device_name(0))


In [ ]:
# 4. Hugging Face login, optional but recommended
from getpass import getpass
from huggingface_hub import login

hf_token = getpass("Hugging Face token, press Enter to skip: ")
if hf_token.strip():
    login(token=hf_token.strip())
    print("Hugging Face login complete")
else:
    print("skip Hugging Face login")


In [ ]:
# 5. Load EXAONE model
import gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct"

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()

print("loaded", MODEL_ID)
print("device", next(model.parameters()).device)


In [ ]:
# 6. Direct model test
messages = [
    {"role": "system", "content": "Answer briefly in Korean."},
    {"role": "user", "content": "EXAONE server test. Reply in one Korean sentence."},
]

input_ids = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to(model.device)

with torch.inference_mode():
    output = model.generate(
        input_ids,
        max_new_tokens=128,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
    )

new_tokens = output[0][input_ids.shape[-1]:]
print(tokenizer.decode(new_tokens, skip_special_tokens=True).strip())


In [ ]:
# 7. Start OpenAI-compatible FastAPI server
import json
import threading
import traceback
import uuid

import nest_asyncio
import torch
import uvicorn
from fastapi import FastAPI
from fastapi.responses import JSONResponse, StreamingResponse
from pydantic import BaseModel, Field

nest_asyncio.apply()

app = FastAPI(title="EXAONE 3.5 7.8B Colab OpenAI-compatible API")

class ChatRequest(BaseModel):
    model: str | None = None
    messages: list[dict]
    temperature: float = 0.1
    max_tokens: int = Field(default=768, alias="max_tokens")
    stream: bool = False


def normalize_messages(messages: list[dict]) -> list[dict]:
    clean_messages = []
    for msg in messages:
        role = msg.get("role", "user")
        if role not in {"system", "user", "assistant"}:
            role = "user"
        clean_messages.append({"role": role, "content": str(msg.get("content", ""))})
    return clean_messages


def build_input_ids(messages: list[dict]):
    clean_messages = normalize_messages(messages)
    try:
        return tokenizer.apply_chat_template(
            clean_messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
        ).to(model.device)
    except Exception:
        rendered = []
        for msg in clean_messages:
            rendered.append(f"[{msg['role']}]\n{msg['content']}")
        rendered.append("[assistant]\n")
        prompt = "\n\n".join(rendered)
        return tokenizer(prompt, return_tensors="pt", truncation=True, max_length=6144)["input_ids"].to(model.device)


def generate_text(req: ChatRequest) -> str:
    input_ids = build_input_ids(req.messages)
    if input_ids.shape[-1] > 6144:
        input_ids = input_ids[:, -6144:]

    max_new_tokens = max(1, min(int(req.max_tokens or 768), 1536))
    temperature = float(req.temperature or 0.0)
    do_sample = temperature > 0

    generation_kwargs = {
        "input_ids": input_ids,
        "max_new_tokens": max_new_tokens,
        "do_sample": do_sample,
        "pad_token_id": tokenizer.pad_token_id or tokenizer.eos_token_id,
        "eos_token_id": tokenizer.eos_token_id,
    }
    if do_sample:
        generation_kwargs["temperature"] = max(temperature, 0.01)
        generation_kwargs["top_p"] = 0.9

    with torch.inference_mode():
        output = model.generate(**generation_kwargs)

    new_tokens = output[0][input_ids.shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


@app.get("/health")
def health():
    return {
        "ok": True,
        "model": MODEL_ID,
        "cuda": torch.cuda.is_available(),
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    }


@app.get("/v1/models")
def models():
    return {"object": "list", "data": [{"id": MODEL_ID, "object": "model"}]}


@app.post("/v1/chat/completions")
def chat(req: ChatRequest):
    try:
        text = generate_text(req)
        completion_id = "chatcmpl-" + uuid.uuid4().hex

        if req.stream:
            def event_stream():
                chunk = {
                    "id": completion_id,
                    "object": "chat.completion.chunk",
                    "model": req.model or MODEL_ID,
                    "choices": [{"index": 0, "delta": {"content": text}, "finish_reason": None}],
                }
                yield "data: " + json.dumps(chunk, ensure_ascii=False) + "\n\n"
                done = {
                    "id": completion_id,
                    "object": "chat.completion.chunk",
                    "model": req.model or MODEL_ID,
                    "choices": [{"index": 0, "delta": {}, "finish_reason": "stop"}],
                }
                yield "data: " + json.dumps(done, ensure_ascii=False) + "\n\n"
                yield "data: [DONE]\n\n"
            return StreamingResponse(event_stream(), media_type="text/event-stream")

        return {
            "id": completion_id,
            "object": "chat.completion",
            "model": req.model or MODEL_ID,
            "choices": [{
                "index": 0,
                "message": {"role": "assistant", "content": text},
                "finish_reason": "stop",
            }],
        }
    except Exception as exc:
        traceback.print_exc()
        return JSONResponse(
            status_code=500,
            content={"error": type(exc).__name__, "message": str(exc), "traceback": traceback.format_exc()},
        )


def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")


server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
print("server started: http://localhost:8000")


In [ ]:
# 8. Test local Colab API
import requests

payload = {
    "model": MODEL_ID,
    "messages": [{"role": "user", "content": "Reply in one Korean sentence. API test."}],
    "temperature": 0.1,
    "max_tokens": 128,
    "stream": False,
}

r = requests.post("http://localhost:8000/v1/chat/completions", json=payload, timeout=180)
print(r.status_code)
print(r.text[:2000])


## Open ngrok public URL

Copy your token from https://dashboard.ngrok.com/get-started/your-authtoken.


In [ ]:
# 9. Open ngrok tunnel
from getpass import getpass
from pyngrok import ngrok
from pyngrok.exception import PyngrokNgrokHTTPError

NGROK_TOKEN = ""  # Optional: paste token here, but do not share the notebook with token saved.

if not NGROK_TOKEN.strip():
    NGROK_TOKEN = getpass("ngrok authtoken: ")

ngrok.kill()
ngrok.set_auth_token(NGROK_TOKEN.strip())

try:
    public_url = ngrok.connect(8000).public_url
except PyngrokNgrokHTTPError as exc:
    print("ngrok failed:", exc)
    print("Close old ngrok endpoints in the ngrok dashboard and run this cell again.")
    raise

base_url = str(public_url).rstrip("/")
print("PUBLIC_URL=", base_url)
print("LLM_API_BASE=", base_url + "/v1")


In [ ]:
# 10. Test public ngrok URL
import requests

headers = {"ngrok-skip-browser-warning": "true"}

r = requests.get(base_url + "/health", headers=headers, timeout=30)
print("health", r.status_code, r.text)

r = requests.post(
    base_url + "/v1/chat/completions",
    headers=headers,
    json={
        "model": MODEL_ID,
        "messages": [{"role": "user", "content": "Reply in one Korean sentence. External connection test."}],
        "temperature": 0.1,
        "max_tokens": 128,
        "stream": False,
    },
    timeout=180,
)
print("chat", r.status_code)
print(r.text[:2000])


## Local PC setup

Put the printed `/v1` URL into `.env`.

```env
LLM_PROVIDER=remote_openai
LLM_MODEL=LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct
LLM_API_BASE=https://xxxxx.ngrok-free.app/v1
LLM_API_KEY=dummy
```

Run locally:

```cmd
conda activate p311_ragreport
cd /d "C:\K-Safety Law RAG"
python web_app.py --host 127.0.0.1 --port 8000
```

CLI check:

```cmd
python cli.py
```
